# Segmentador de gases en Google Colab

Simulador de fugas en imagen térmica (LWIR) y detector clásico, con la verdad de terreno conocida.
Repositorio: https://github.com/eneko1982/Segmentador_gases

**Cómo usarlo:** ejecuta las celdas en orden (menú *Entorno de ejecución → Ejecutar todo*, o `Ctrl+F9`).
La celda 6 es el panel de control completo (ajustes, vídeo con máscaras, resultados y exportación).

## 1. Instalar

In [ ]:
import os
if os.path.basename(os.getcwd()) == "Segmentador_gases":      # por si la celda se ejecuta otra vez
    os.chdir("..")
if os.path.exists("Segmentador_gases"):
    !git -C Segmentador_gases pull -q                          # actualiza la copia que ya había
else:
    !git clone -q https://github.com/eneko1982/Segmentador_gases
os.chdir("Segmentador_gases")
print("Carpeta de trabajo:", os.getcwd())
!pip install -q -r requirements.txt

## 2. Simular una fuga

Cambia los parámetros a tu gusto. La lista completa está en `docs/parametros.md`; los presets (`hot_plate_dT48`,
`hot_plate_dT5`, `cold_sky`, `long_turbulent`, `intermittent_leak`) están en la carpeta `configs/`.
La media resolución es unas 4 veces más rápida.

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import ogi_sim as sim

PRESET = "hot_plate_dT5"          # escena de partida
CAMBIOS = {                       # cualquier parámetro de docs/parametros.md
    "flow_g_s": 0.03,             # caudal (g/s)
    "wind_ms": 0.3,               # viento (m/s, + = hacia la derecha)
    "turb_rms": 0.15,             # turbulencia (m/s)
    "duration_s": 5.0,
    "leak_on_s": 2.0,
}
COMPLETA = False                  # True = 384x288 px (más lento)

cfg = sim.apply_overrides(sim.Config(), {**sim.load_preset(PRESET), **CAMBIOS})
if not COMPLETA:
    cfg = sim.half_resolution(cfg)
seq = sim.simulate(cfg)
meta = json.loads(seq["meta"])
fps = meta["fps"]
print(f"{len(seq['u8'])} fotogramas de {cfg.width}x{cfg.height}; gas emitido: {meta['emitted_g']:.3f} g; "
      f"cambio máximo en un píxel: {1e3 * float(np.abs(seq['contrast']).max()):.0f} mK")

## 3. Ver la secuencia

Arriba, lo que daría la cámara (8 bit). Abajo, el cambio causado por el gas, sin ruido (en mK).

In [ ]:
tiempos = [1.0, 2.5, 3.5, 4.9]
fig, ax = plt.subplots(2, len(tiempos), figsize=(4 * len(tiempos), 6.5))
for j, t in enumerate(tiempos):
    i = min(int(t * fps), len(seq["u8"]) - 1)
    ax[0, j].imshow(seq["u8"][i], cmap="gray", vmin=0, vmax=255)
    ax[0, j].set_title(f"t = {i / fps:.1f} s")
    im = ax[1, j].imshow(seq["contrast"][i].astype(float) * 1e3, cmap="RdBu_r", vmin=-100, vmax=100)
    for a in ax[:, j]:
        a.set_xticks([]); a.set_yticks([])
fig.colorbar(im, ax=ax[1, :], label="mK", shrink=0.8)
plt.show()

## 4. Ejecutar el detector

El contorno **naranja** es lo que marca el detector. El **azul** es la región real (simulador) con **el mismo
umbral**: el propio del detector (2,5 sigma del ruido tras el filtrado).

In [ ]:
from gas_detect import alarm, detect, preview_frames, scores

res = detect(seq["u8"], fps, gas_free_s=0.8 * cfg.leak_on_s)
pred, valid = res["mask"], res["valid"]
unit_k = meta["k_per_dn"]                                   # K por nivel de gris
on = int(round(cfg.leak_on_s * fps))
contrast = np.abs(seq["contrast"].astype(np.float32))
last = slice(len(pred) - int(fps), len(pred))               # último segundo
umbral_k = 2.5 * res["sigma"] * unit_k
verdad = (contrast > umbral_k) & valid
sc = scores(pred[last], verdad[last])
retardo = np.flatnonzero(alarm(pred)[on:])
print(f"Ruido tras el filtrado : {1e3 * res['sigma'] * unit_k:.1f} mK (un píxel: {1e3 * cfg.netd_k:.0f} mK)")
print(f"Falsa alarma antes     : {'SÍ' if alarm(pred)[res['n_free']:on].any() else 'no'}")
print(f"Retardo de detección   : {'no detectado' if not len(retardo) else f'{retardo[0] / fps:.2f} s'}")
print(f"Máscara frente a la verdad a {1e3 * umbral_k:.0f} mK: IoU {sc['iou']:.2f}, recall {sc['recall']:.2f}, "
      f"precisión {sc['precision']:.2f}")

frame = preview_frames(seq["u8"], pred, verdad, step=len(pred) - 1)[-1]    # último fotograma
plt.figure(figsize=(6, 4.5)); plt.imshow(frame); plt.axis("off"); plt.title("Naranja: detector · Azul: verdad")
plt.show()

## 5. Comparar detector y verdad con el mismo umbral

Simula la escena de ΔT = 5 K con tres caudales y dibuja las dos líneas, a 1× NETD y al umbral del detector.
Tarda aproximadamente medio minuto.

In [ ]:
import make_figure

fig = make_figure.build_figure("hot_plate_dT5", [0.003, 0.03, 0.3])
plt.show()

## 6. El panel completo, dentro del cuaderno

Es el mismo panel de control de la aplicación web, **sin servidor ni túnel**, así que funciona también en redes
corporativas que bloquean esos servicios. Ejecuta la celda y usa las pestañas:

- **⚙️ Ajustes**: tres bloques (vídeo y cámara, simulación del gas, cálculos del detector) con todos los parámetros
  (los menos habituales están en «Más parámetros»). Elige un preset si quieres.
- Pulsa **Simular y calcular** (arriba). La vista rápida usa media resolución; elige «Completa» para el tamaño real.
- **🎬 Vídeo**: reproductor con una, dos o tres vistas. Activa o desactiva la **máscara real** (azul) y la
  **detectada** (naranja), cambia entre contorno, relleno y acierto/error, y elige con qué umbral se define cada máscara.
- **📊 Resultados**: métricas, tabla por umbrales, series temporales y galería de fotogramas.
- **📦 Exportar**: CSV, JSON, imágenes, GIF, informe HTML y un ZIP. Los archivos quedan en la carpeta `salida/`
  (panel de archivos de la izquierda de Colab) y salen botones para descargarlos.

Si el reproductor tarda o no carga, elige «Mostrar: 1 de cada 4» y menos vistas.

In [ ]:
import notebook_ui
panel = notebook_ui.launch()